In [7]:
# Homework 4
# Center of Mass Position and Velocity
# Maritza Torres

In [8]:
# remember this is just a template,
# you don't need to follow every step.
# If you have your own method to solve the homework,
# it is totally fine

In [9]:
# import modules
import numpy as np # import numpy
import astropy.units as u # import astropy units
from astropy import constants as const # import astropy constants
from ReadFile import Read
 # this is so we can read the file and take the data for our calculations
import astropy.table as tbl # make a table

In [28]:
class CenterOfMass:
# Class to define COM position and velocity properties 
# of a given galaxy and simulation snapshot

    def __init__(self, filename, ptype):
        ''' Class to calculate the 6-D phase-space position of a galaxy's center of mass using
        a specified particle type. 
            
            PARAMETERS
            self: allows all information to be store and produce its own data
            filename : `str` is the file the data will come from to make the calculations
            ptype : `int; 1, 2, or 3` is the type of particle (Type 1 = Dark Matter, Type 2 = Disk Stars, Type 3 = Bulge Stars)
        '''
     
        # read data in the given file using Read
        self.time, self.total, self.data = Read(filename)                                                                                             

        #create an array to store indexes of particles of desired Ptype                                
        self.index = np.where(self.data['type'] == ptype)

        # stores the mass, positions, velocities of only the particles of the given type
        # the following only gives the example of storing the mass and with its units.
        self.m = self.data['m'][self.index] 
        # write your own code to complete this for positions and velocities
        self.x = self.data['x'][self.index] 
        self.y = self.data['y'][self.index] 
        self.z = self.data['z'][self.index] 
        self.vx = self.data['vx'][self.index] 
        self.vy = self.data['vy'][self.index] 
        self.vz = self.data['vz'][self.index] 


    def COMdefine(self,a,b,c,m):
        ''' Method to compute the COM of a generic vector quantity by direct weighted averaging.
        
        PARAMETERS
        self  allows all information to be store and produce its own data
        a : `float or np.ndarray of floats`first vector component
        b : `float or np.ndarray of floats` second vector component
        c : `float or np.ndarray of floats` third vector component
        m : `float or np.ndarray of floats` particle masses
        
        RETURNS
        -------
        a_com : `float` first component on the COM vector
        b_com : `float` second component on the COM vector
        c_com : `float` third component on the COM vector
        '''
        # write your own code to compute the generic COM 
        #using Eq. 1 in the homework instructions
        # xcomponent Center of mass
        a_com = np.sum(a*m)/np.sum(m) # this is the x component of mass
        # ycomponent Center of mass  
        b_com = np.sum(b*m)/np.sum(m) # this is the y component of mass
        # zcomponent Center of mass
        c_com = np.sum(c*m)/np.sum(m) # this is the z component of mass
        
        # return the 3 components separately
        return a_com, b_com, c_com # return the components by using commas
    
    
    def COM_P(self, delta):
        '''Method to compute the position of the center of mass of the galaxy 
        using the shrinking-sphere method.

        PARAMETERS
        self  allows all information to be store and produce its own data
        delta : `float, optional` error tolerance in kpc. Default is 0.1 kpc
        
        RETURNS
        ----------
        p_COM : `np.ndarray of astropy.Quantity' 3-D position of the center of mass in kpc
        '''                                                                     

        # Center of Mass Position                                                                                      
        ###########################                                                                                    

        # Try a first guess at the COM position by calling COMdefine                                                   
        x_COM, y_COM, z_COM = self.COMdefine(self.x, self.y, self.z, self.m)
        # compute the magnitude of the COM position vector.
        # write your own code below
        r_COM = np.sqrt((x_COM)**2+ (y_COM)**2+ (z_COM)**2) # this is the magnitude of the position vector


        # iterative process to determine the center of mass                                                            

        # change reference frame to COM frame                                                                          
        # compute the difference between particle coordinates                                                          
        # and the first guess at COM position
        # write your own code below
        x_new = self.x - x_COM # this is the new x
        y_new = self.y - y_COM # this is the new y
        z_new = self.z - z_COM # this is the new z
        r_new = np.sqrt((x_new)**2 + (y_new)**2 + (z_new)**2)

        # find the max 3D distance of all particles from the guessed COM                                               
        # will re-start at half that radius (reduced radius)                                                           
        r_max = max(r_new)/2.0
        
        # pick an initial value for the change in COM position                                                      
        # between the first guess above and the new one computed from half that volume
        # it should be larger than the input tolerance (delta) initially
        change = 1000.0

        # start iterative process to determine center of mass position                                                 
        # delta is the tolerance for the difference in the old COM and the new one.    
        
        while (change > delta):
            # select all particles within the reduced radius (starting from original x,y,z, m)
            # write your own code below (hints, use np.where)
            index2 = np.where(r_new < r_max)
            x2 = self.x[index2]
            y2 = self.y[index2]
            z2 = self.z[index2]
            m2 = self.m[index2]

            # Refined COM position:                                                                                    
            # compute the center of mass position using                                                                
            # the particles in the reduced radius
            # write your own code below
            x_COM2, y_COM2, z_COM2 = self.COMdefine(x2,y2,z2,m2)
            # compute the new 3D COM position
            # write your own code below
            r_COM2 = np.sqrt((x_COM2)**2 + (y_COM2)**2 + (z_COM2)**2)

            # determine the difference between the previous center of mass position                                    
            # and the new one.                                                                                         
            change = np.abs(r_COM - r_COM2)
            # uncomment the following line if you want to check this                                                                                               
            print ("CHANGE = ", change)                                                                                     

            # Before loop continues, reset : r_max, particle separations and COM                                        

            # reduce the volume by a factor of 2 again                                                                 
            r_max /= 2.0
            # check this.                                                                                              
            print ("maxR", r_max)                                                                                      

            # Change the frame of reference to the newly computed COM.                                                 
            # subtract the new COM
            # write your own code below
            x_new = x2 - x_COM2 # this is the new z using the index2
            y_new = y2 - y_COM2 # this is the new z
            z_new = z2 - z_COM2 # this is the new z
            r_new = np.sqrt((x_new)**2 + (y_new)**2 + (z_new)**2) # this is the new magntitude

            # set the center of mass positions to the refined values                                                   
            x_COM = x_COM2
            y_COM = y_COM2
            z_COM = z_COM2
            r_COM = r_COM2

            # create an array (np.array) to store the COM position                                                                                                                                                       
            p_COM = np.array([x_COM, y_COM, z_COM]) 

        # set the correct units using astropy and round all values
        # and then return the COM positon vector
        # write your own code below
            p_COM = np.round(p_COM,2)* u.kpc 
        return p_COM
        
        
        
    def COM_V(self, x_COM, y_COM, z_COM):
        ''' Method to compute the center of mass velocity based on the center of mass
        position.

        PARAMETERS
        self  allows all information to be store and produce its own data
        x_COM : 'astropy quantity' The x component of the center of mass in kpc
        y_COM : 'astropy quantity' The y component of the center of mass in kpc
        z_COM : 'astropy quantity' The z component of the center of mass in kpc
            
        RETURNS
        -------
        v_COM : `np.ndarray of astropy.Quantity' 3-D velocity of the center of mass in km/s
        '''
        
        # the max distance from the center that we will use to determine 
        #the center of mass velocity                   
        rv_max = 15.0*u.kpc

        # determine the position of all particles relative to the center of mass position (x_COM, y_COM, z_COM)
        # write your own code below
        
        xV = self.x*u.kpc - x_COM # need to make sure position has correct units 
        yV = self.y*u.kpc - y_COM  
        zV = self.z*u.kpc - z_COM  
        rV = np.sqrt((xV)**2 + (yV)**2 + (zV)**2)
        
        # determine the index for those particles within the max radius
        # write your own code below
        indexV = np.where(rV < rv_max)
        
        # determine the velocity and mass of those particles within the mas radius
        # write your own code below
        # Note that x_COM, y_COM, z_COM are astropy quantities and you can only subtract one astropy quantity from another
        # So, when determining the relative positions, assign the appropriate units to self.x
        vx_new = self.vx[indexV]  
        vy_new = self.vy[indexV]  
        vz_new = self.vz[indexV]  
        m_new =  self.m[indexV] 
        
        # compute the center of mass velocity using those particles
        # write your own code below
        vx_COM, vy_COM, vz_COM = self.COMdefine(vx_new,vy_new,vz_new,m_new)
        
        # create an array to store the COM velocity
        # write your own code below
        v_COM = np.array([vx_COM, vy_COM, vz_COM]) 
        
        # return the COM vector
        # set the correct units using astropy
        # round all values                                                                                        
        
        v_COM= np.round(v_COM,2) * u.km/u.s  
        return v_COM

In [29]:
# Create a Center of mass object for the MW, M31 and M33
# below is an example of using the class for MW
MW_COM = CenterOfMass("MW_000.txt", 2)

In [30]:
# below gives you an example of calling the class's functions
# MW:   store the position and velocity COM
MW_COM_p = MW_COM.COM_P(0.1)
print(MW_COM_p)
MW_COM_v = MW_COM.COM_V(MW_COM_p[0], MW_COM_p[1], MW_COM_p[2])
print(MW_COM_v)

CHANGE =  0.22978104414074885
maxR 11.020481333398111
CHANGE =  0.2599205188628648
maxR 5.5102406666990555
CHANGE =  0.0006566884144847407
maxR 2.7551203333495278
[-0.87  2.39 -1.42] kpc
[-0.47  3.41 -1.33] km / s


In [50]:
# now write your own code to answer questions
# question 1
MW_COM = CenterOfMass("MW_000.txt", 2) # this is the file for MW
M31_COM = CenterOfMass("M31_000.txt", 2) # this is the file for M31
M33_COM = CenterOfMass("M33_000.txt", 2) # this is the file for M33
#MW COM position and velocity in kpc and km/s
MW_COM_p = MW_COM.COM_P(0.1) # use tolerance of 0.1 kpc
print("MW Position: ", MW_COM_p) # print the velocity for MW
MW_COM_v = MW_COM.COM_V(MW_COM_p[0], MW_COM_p[1], MW_COM_p[2])
print("MW Velocity ", MW_COM_v) # print the velocity for MW
#M31 COM position and velocity in kpc and km/s
M31_COM_p = M31_COM.COM_P(0.1) # use tolerance of 0.1 kpc
print("M31 Position: ", M31_COM_p) # print the positions for M31
M31_COM_v = M31_COM.COM_V(M31_COM_p[0], M31_COM_p[1], M31_COM_p[2])
print("M31 Velocity ", M31_COM_v) # print the velocity for M31
#M33 COM position and velocity in kpc and km/s
M33_COM_p = M33_COM.COM_P(0.1) # use tolerance of 0.1 kpc
print("M33 Position: ", M33_COM_p) # print the positions for M33
M33_COM_v = M33_COM.COM_V(M33_COM_p[0], M33_COM_p[1], M33_COM_p[2])
print("M33 Velocity ", M33_COM_v) # print the velocity for M33

CHANGE =  0.22978104414074885
maxR 11.020481333398111
CHANGE =  0.2599205188628648
maxR 5.5102406666990555
CHANGE =  0.0006566884144847407
maxR 2.7551203333495278
MW Position:  [-0.87  2.39 -1.42] kpc
MW Velocity  [-0.47  3.41 -1.33] km / s
CHANGE =  0.04459429679900495
maxR 16.68229943877858
M31 Position:  [-377.66  611.43 -284.64] kpc
M31 Velocity  [ 72.85 -72.14  49.  ] km / s
CHANGE =  0.01308865872044862
maxR 3.9330197268235163
M33 Position:  [-476.22  491.44 -412.4 ] kpc
M33 Velocity  [ 44.42 101.78 142.23] km / s


In [53]:
# question 2 
MW31_sep1 = np.sqrt(np.sum(((MW_COM_p) - (M31_COM_p))**2)) # we are trying to get the magnitude so sqrt what we got from the last question
MW31_sep = np.round(MW31_sep1,3) # we must round 
print("Magnitude Separation MW-M31 (kpc)", MW31_sep) # print the separation
MW = np.sqrt((MW_COM_v[0]**2) + (MW_COM_v[1]**2) + (MW_COM_v[2]**2)) # approximate velocity as I got a higher solution using my last method
M31 = np.sqrt((M31_COM_v[0]**2) + (M31_COM_v[1]**2) + (M31_COM_v[2]**2)) # sqrt everything using the soltuion from the last question for MW and M31
MY = abs(MW-M31) # absolute the difference
MW31_vel = np.round(MY,3) # round
print("Velocity Separation MW-M31 (km/s)", MW31_vel) # print the velocity difference

Magnitude Separation MW-M31 (kpc) 770.139 kpc
Velocity Separation MW-M31 (km/s) 109.942 km / s


In [58]:
# question 3
M3331_sep1 = np.sqrt(np.sum(((M33_COM_p) - (M31_COM_p))**2)) # we are trying to get the magnitude so sqrt what we got from the 1st question
M3331_sep = np.round(M3331_sep1,3) # we must round 
print("Magnitude Separation M33-M31 (kpc)", M3331_sep) # print the separation
M33 = np.sqrt((M33_COM_v[0]**2) + (M33_COM_v[1]**2) + (M33_COM_v[2]**2)) # approximate velocity  
M312 = np.sqrt((M31_COM_v[0]**2) + (M31_COM_v[1]**2) + (M31_COM_v[2]**2)) # sqrt everything using the soltuion from the 1st question for M33 and M31
MY2 = abs(M33-M312) # absolute the difference
M3331_vel = np.round(MY2,3) # round
print("Velocity Separation M33-M31 (km/s)", M3331_vel) # print the velocity difference

Magnitude Separation M33-M31 (kpc) 201.083 kpc
Velocity Separation M33-M31 (km/s) 66.816 km / s


In [59]:
# question 4
# The iterative process to determine the center of mass is important since it provides you more accurate/stable calculations, 
# which are vital when studying galaxies that are potentially merging together. It gets rid of results that could potentially cause errors when 
# making calculations. Overall it gives better evidence on the development of the galaxies evolving together over time as they come together. 
# Basically providing a stable foundation for when the merge occurs which would allow astronomers to predict the dynamics and calculations more 
# accurately with credibility.